# RPD – PhenoBench Baseline B0 Multi-Scenario Training (Kaggle)

Repo: [`ndhoang2310/RPD_Reproduce`](https://github.com/ndhoang2310/RPD_Reproduce)

| Mục | Cấu hình |
|---|---|
| Accelerator | **GPU T4 x1** hoặc **GPU T4 x2** |
| Internet | **ON** |
| Dataset | `ndhoang2310/phenobench-dataset` |

### Danh mục 7 Kịch bản B0:
- `B0_run1_anchor` : Mốc chuẩn sạch (Poly, LR 1e-4, Batch 4, FP32)
- `B0_run2_cosine` : Khảo sát Scheduler (Cosine Annealing, LR 1e-4, Batch 4, FP32)
- `B0_run3_lr2e4`  : Khảo sát LR (Poly, LR 2e-4, Batch 4, FP32)
- `B0_run4_lr3e4`  : Khảo sát LR giới hạn (Poly, LR 3e-4, Batch 4, FP32)
- `B0_run5_cosine_lr2e4` : Kết hợp Scheduler + LR (Cosine, LR 2e-4, Batch 4, FP32)
- `B0_run6_batch8_fp16_poly` : Khảo sát Phần cứng (Poly, LR 1e-4, Batch 8, FP16)
- `B0_run7_batch8_fp16_cosine` : Phần cứng + Scheduler (Cosine, LR 1e-4, Batch 8, FP16)

In [ ]:
# ================================================================
# Ô 1 — Cài đặt dependencies chuẩn hóa
# Khắc phục triệt để các lỗi: setuptools<70, tensorboard, pandas
# ================================================================
!nvidia-smi
!pip install -q \
    "pytorch-lightning==1.9.5" \
    "torchmetrics==0.11.4" \
    "setuptools<70" \
    tensorboard \
    matplotlib pandas tabulate \
    timm scikit-image tifffile albumentations

In [ ]:
# ================================================================
# Ô 2 — Chọn kịch bản và Bắt đầu Huấn luyện
# ================================================================
import os, glob, yaml, shutil

# >>> CHỌN KỊCH BẢN BẠN ĐƯỢC PHÂN CÔNG TẠI ĐÂY <<<
SCENARIO = "B0_run1_anchor"
# Lựa chọn: ["B0_run1_anchor", "B0_run2_cosine", "B0_run3_lr2e4", "B0_run4_lr3e4", "B0_run5_cosine_lr2e4", "B0_run6_batch8_fp16_poly", "B0_run7_batch8_fp16_cosine"]

REPO_URL = "https://github.com/ndhoang2310/RPD_Reproduce.git"
REPO_DIR = "/kaggle/working/RPD"
LOG_DIR  = f"/kaggle/working/log_dir/{SCENARIO}"

# ── 1. Chuẩn bị repository ──────────────────────────────────────
if not os.path.exists(os.path.join(REPO_DIR, "multi_metric_train.py")):
    !rm -rf {REPO_DIR}
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

# ── 2. Tự động tìm dataset PhenoBench ───────────────────────────
DATASET_CANDIDATES = [
    "/kaggle/input/datasets/ndhoang2310/phenobench-dataset/PhenoBench",
    "/kaggle/input/phenobench-dataset/PhenoBench",
    "/kaggle/input/datasets/ndhoang2310/phenobench-dataset",
    "/kaggle/input/phenobench-dataset",
]
dataset_dir = next(
    (p for p in DATASET_CANDIDATES if os.path.exists(os.path.join(p, "train", "images"))),
    None
)
if dataset_dir is None:
    print("[ERROR] Không tìm thấy PhenoBench! Vui lòng gắn dataset 'ndhoang2310/phenobench-dataset'.")
    !find /kaggle/input -maxdepth 3 -type d
    raise FileNotFoundError("Chưa gắn dataset PhenoBench.")
print(f"[OK] Dataset: {dataset_dir}")
print(f"     Số ảnh train: {len(os.listdir(os.path.join(dataset_dir, 'train', 'images')))}")

# ── 3. Tải config của kịch bản đã chọn ──────────────────────────
cfg_path = f"./configs/b0_scenarios/{SCENARIO}.yaml"
if not os.path.exists(cfg_path):
    raise FileNotFoundError(f"Không tìm thấy cấu hình {cfg_path}!")

with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# Gán đường dẫn dataset chính xác
cfg["data"]["path_to_dataset"] = dataset_dir
active_cfg_path = f"/kaggle/working/active_config_{SCENARIO}.yaml"
with open(active_cfg_path, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f"[OK] Kịch bản: {SCENARIO}")
print(f"     Scheduler: {cfg['train'].get('scheduler', 'poly')} | LR: {cfg['train'].get('learning_rate')} | Batch: {cfg['train'].get('batch_size')} | Precision: {cfg['train'].get('precision', 32)}")

# ── 4. Tự động Resume nếu có Checkpoint cũ ──────────────────────
ckpt_candidates = glob.glob(f"{LOG_DIR}/**/checkpoints/last.ckpt", recursive=True) + glob.glob("/kaggle/input/**/checkpoints/last.ckpt", recursive=True)
resume_flag = f"--ckpt_path {ckpt_candidates[0]} --resume" if ckpt_candidates else ""
if ckpt_candidates:
    print(f"[RESUME] Tiếp tục huấn luyện từ: {ckpt_candidates[0]}")
else:
    print("[TRAIN] Huấn luyện từ đầu (Epoch 0 -> 100)")

# ── 5. Bắt đầu Train ───────────────────────────────────────────
!python multi_metric_train.py \
    --config {active_cfg_path} \
    --export_dir {LOG_DIR} \
    --devices 2 \
    {resume_flag}

In [ ]:
# ================================================================
# Ô 3 — Hiển thị Đồ thị Validation và Báo cáo Kết quả Trực tiếp
# ================================================================
from IPython.display import Image, display

curves_image = os.path.join(LOG_DIR, "val_curves.png")
if os.path.exists(curves_image):
    print("\n=== ĐỒ THỊ TIẾN TRÌNH VALIDATION (4 BIỂU ĐỒ) ===")
    display(Image(filename=curves_image))
else:
    print("[Notice] Chưa có file val_curves.png. Chạy lệnh xuất đồ thị thủ công:")
    !python utils/plot_val_curves.py --log_dir {LOG_DIR}
    if os.path.exists(curves_image):
        display(Image(filename=curves_image))

summary_txt = os.path.join(LOG_DIR, "val_summary.txt")
if os.path.exists(summary_txt):
    with open(summary_txt, "r", encoding="utf-8") as f:
        print(f.read())

In [ ]:
# ================================================================
# Ô 4 — Nén Kết quả để Tải về máy tính 1-Click
# ================================================================
import shutil

zip_filename = f"/kaggle/working/{SCENARIO}_results"
shutil.make_archive(zip_filename, 'zip', LOG_DIR)
print(f"[THÀNH CÔNG] File kết quả đã sẵn sàng để tải về:")
print(f"👉 {zip_filename}.zip ({round(os.path.getsize(zip_filename + '.zip') / 1024 / 1024, 2)} MB)")
print("Tải file này về máy để gộp so sánh cùng nhóm!")